# Ungraded Lab: Feature Selection with Titanic Dataset

Feature selection involves picking the set of features that are most relevant to the target variable.
In this notebook you will run through different feature selection techniques on the [Titanic Dataset](https://www.kaggle.com/c/titanic) using a **Random Forest Classifier**.

## Setup

In [ ]:
import os, seaborn as sns
os.makedirs('./data', exist_ok=True)
titanic_raw = sns.load_dataset('titanic')
titanic_raw.to_csv('./data/titanic.csv', index=False)
print(f"Saved: ./data/titanic.csv  shape={titanic_raw.shape}")

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, SelectFromModel, f_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt

## Load the Dataset

In [ ]:
df = pd.read_csv('./data/titanic.csv')
print(df.dtypes)
df.describe(include='all')

In [ ]:
df.head()

## Remove Unwanted Features

The seaborn Titanic dataset contains derived/duplicate columns. Remove `class`, `who`, `adult_male`, `deck`, `embark_town`, `alive`, `alone`.

In [ ]:
df.isna().sum()

In [ ]:
columns_to_remove = ['class','who','adult_male','deck','embark_town','alive','alone']
df.drop(columns_to_remove, axis=1, inplace=True)
df.head()

## Handle Missing Values

In [ ]:
df['age'].fillna(df['age'].median(), inplace=True)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
print(df.isna().sum())

## Encode Categorical Features

In [ ]:
df['sex'] = df['sex'].map({'male':1,'female':0})
le = LabelEncoder()
df['embarked'] = le.fit_transform(df['embarked'])
df.head()

## Model Performance

Split into features `X` and target `Y`, then compare feature selection methods using a **RandomForestClassifier**.

In [ ]:
X = df.drop('survived', axis=1)
Y = df['survived']

### Fit the Model and Calculate Metrics

In [ ]:
def fit_model(X, Y):
    model = RandomForestClassifier(criterion='entropy', random_state=47)
    model.fit(X, Y)
    return model

In [ ]:
def calculate_metrics(model, X_test_scaled, Y_test):
    y_pred = model.predict(X_test_scaled)
    return (accuracy_score(Y_test, y_pred), roc_auc_score(Y_test, y_pred),
            precision_score(Y_test, y_pred), recall_score(Y_test, y_pred),
            f1_score(Y_test, y_pred))

In [ ]:
def train_and_get_metrics(X, Y):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=123)
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s  = scaler.transform(X_test)
    model = fit_model(X_train_s, Y_train)
    return calculate_metrics(model, X_test_s, Y_test)

In [ ]:
def evaluate_model_on_features(X, Y):
    acc, roc, prec, rec, f1 = train_and_get_metrics(X, Y)
    return pd.DataFrame([[acc, roc, prec, rec, f1, X.shape[1]]],
                        columns=['Accuracy','ROC','Precision','Recall','F1 Score','Feature Count'])

In [ ]:
all_features_eval_df = evaluate_model_on_features(X, Y)
all_features_eval_df.index = ['All features']
results = all_features_eval_df
results

## Correlation Matrix

In [ ]:
plt.figure(figsize=(12,10))
cor = df.corr()
import seaborn as sns
sns.heatmap(cor, annot=True, cmap=plt.cm.PuBu)
plt.show()

## Filter Methods

### Correlation with the Target Variable

In [ ]:
cor_target = abs(cor['survived'])
relevant_features = cor_target[cor_target > 0.2]
names = [index for index, value in relevant_features.items()]
names.remove('survived')
print(names)

In [ ]:
strong_features_eval_df = evaluate_model_on_features(df[names], Y)
strong_features_eval_df.index = ['Strong features']
results = pd.concat([results, strong_features_eval_df])
results

### Correlation with Other Features

In [ ]:
plt.figure(figsize=(10,8))
new_corr = df[names].corr()
sns.heatmap(new_corr, annot=True, cmap=plt.cm.Blues)
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[names+['survived']].corr(), annot=True, cmap=plt.cm.Blues)
plt.title('Correlation Matrix of Strong Features')
plt.show()

In [ ]:
upper = new_corr.where(np.triu(np.ones(new_corr.shape), k=1).astype(bool))
highly_correlated = [col for col in upper.columns if any(upper[col].abs() > 0.7)]
print(f'Highly correlated features (>0.7): {highly_correlated}')
subset_feature_corr_names = [x for x in names if x not in highly_correlated]
print(f'Subset features: {subset_feature_corr_names}')

In [ ]:
subset_feature_eval_df = evaluate_model_on_features(df[subset_feature_corr_names], Y)
subset_feature_eval_df.index = ['Subset features']
results = pd.concat([results, subset_feature_eval_df])
results

### Univariate Selection with Sci-Kit Learn

Select top 5 features using `SelectKBest` with ANOVA F-test.

In [ ]:
def univariate_selection():
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=123)
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    selector = SelectKBest(f_classif, k=5)
    selector.fit_transform(X_train_s, Y_train)
    feature_idx = selector.get_support()
    for name, included in zip(df.drop('survived', axis=1).columns, feature_idx):
        print(f'{name}: {included}')
    return df.drop('survived', axis=1).columns[feature_idx]

univariate_feature_names = univariate_selection()

In [ ]:
univariate_eval_df = evaluate_model_on_features(df[univariate_feature_names], Y)
univariate_eval_df.index = ['F-test']
results = pd.concat([results, univariate_eval_df])
results

## Wrapper Methods

### Recursive Feature Elimination

In [ ]:
RFE?

In [ ]:
def run_rfe():
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=123)
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    model = RandomForestClassifier(criterion='entropy', random_state=47)
    rfe = RFE(model, n_features_to_select=5)
    rfe.fit(X_train_s, Y_train)
    return df.drop('survived', axis=1).columns[rfe.get_support()]

rfe_feature_names = run_rfe()

In [ ]:
rfe_eval_df = evaluate_model_on_features(df[rfe_feature_names], Y)
rfe_eval_df.index = ['RFE']
results = pd.concat([results, rfe_eval_df])
results

## Embedded Methods

### Feature Importances

In [ ]:
def feature_importances_from_tree_based_model_():
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=123)
    scaler = StandardScaler().fit(X_train)
    model = RandomForestClassifier()
    model.fit(scaler.transform(X_train), Y_train)
    plt.figure(figsize=(10,6))
    pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).plot(kind='barh')
    plt.title('Random Forest Feature Importances - Titanic')
    plt.show()
    return model

def select_features_from_model(model):
    sfm = SelectFromModel(model, prefit=True, threshold=0.05)
    return df.drop('survived', axis=1).columns[sfm.get_support()]

model = feature_importances_from_tree_based_model_()
feature_imp_feature_names = select_features_from_model(model)

In [ ]:
feat_imp_eval_df = evaluate_model_on_features(df[feature_imp_feature_names], Y)
feat_imp_eval_df.index = ['Feature Importance']
results = pd.concat([results, feat_imp_eval_df])
results

### L1 Regularization

In [ ]:
def run_l1_regularization():
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=123)
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    selection = SelectFromModel(LinearSVC(C=1, penalty='l1', dual=False, max_iter=5000))
    selection.fit(X_train_s, Y_train)
    return df.drop('survived', axis=1).columns[selection.get_support()]

l1reg_feature_names = run_l1_regularization()

In [ ]:
l1reg_eval_df = evaluate_model_on_features(df[l1reg_feature_names], Y)
l1reg_eval_df.index = ['L1 Reg']
results = pd.concat([results, l1reg_eval_df])
results

## Summary

With these results you can decide which set of features to use. `sex` and `pclass` are consistently among the most important features — reflecting the 'women and children first' evacuation policy and the advantage of higher-class accommodations.